# 🛡️ Governance & Security for CAMEL Agents with TealTiger

This cookbook demonstrates how to add **deterministic governance** to CAMEL-AI multi-agent systems using [`camelai-tealtiger`](https://pypi.org/project/camelai-tealtiger/). You'll learn how to:

1. **Observe agents** in zero-config mode — track cost, detect PII, build behavioral baselines
2. **Enforce policies** — block unauthorized steps before they execute
3. **Detect PII** — automatically identify emails, SSNs, and other sensitive data in agent messages
4. **Kill switch** — freeze/unfreeze individual agents mid-conversation
5. **Validate roles** — restrict which agent roles can operate in your system
6. **Generate baselines** — capture normal agent behavior for anomaly detection
7. **Audit with TEEC** — structured evidence contract for compliance and observability

**Key property:** All governance evaluation is deterministic — no LLM in the governance path. Same input always produces the same decision. Adds <2ms latency per evaluation.

## Why governance for multi-agent systems?

In CAMEL-AI's multi-agent architecture, agents interact autonomously — calling tools, exchanging messages, and making decisions. In production, you need guarantees:

- Agents only perform authorized actions
- Sensitive data (PII) is detected and flagged before leaving the system
- Misbehaving agents can be halted instantly without stopping the entire system
- Every governance decision is auditable with structured evidence

TealTiger provides these guarantees with zero external dependencies and sub-millisecond evaluation.

## Installation

In [ ]:
!pip install camelai-tealtiger -q

## Zero-Config Mode

Add governance with **zero configuration**. `TealTigerAgentHook` observes all agent steps, tracks cost, detects PII, and allows everything through — producing structured audit entries for observability.

This is the simplest way to start: attach the hook, run your agents, and inspect what happened.

In [ ]:
from camelai_tealtiger import TealTigerAgentHook

# Create hook — no configuration needed
hook = TealTigerAgentHook()

# Before each agent step — governance evaluates and returns a decision
decision = hook.pre_step(
    agent_id="assistant-001",
    step_content="Summarize the document about AI safety regulations...",
    tool_name="web_search",
    tool_args={"query": "AI safety regulations 2025"},
    agent_role="researcher",
)

print(f"Action: {decision['action']}")          # ALLOW (zero-config allows everything)
print(f"Cost tracked: ${decision['cost_tracked']:.6f}")
print(f"PII detected: {len(decision['pii_detected'])} items")
print(f"Correlation ID: {decision['correlation_id'][:8]}...")

# After each agent step — record token usage for cost tracking
hook.post_step(
    agent_id="assistant-001",
    step_result="Found 3 relevant papers on AI alignment and safety frameworks...",
    token_usage={"prompt_tokens": 200, "completion_tokens": 150},
)

# View session summary
print("\n--- Session Summary ---")
for agent_id, summary in hook.summary.items():
    print(f"{agent_id}: {summary.step_count} steps, ${summary.total_cost:.6f}")

## PII Detection

TealTiger automatically scans agent step content for personally identifiable information (PII) — emails, SSNs, phone numbers, and more. Detection happens **in-process** with zero external calls.

In [ ]:
from camelai_tealtiger import TealTigerAgentHook

hook = TealTigerAgentHook()

# Step content containing PII
decision = hook.pre_step(
    agent_id="data-agent",
    step_content=(
        "The customer's email is john.doe@example.com and their "
        "SSN is 123-45-6789. Please update the records."
    ),
    tool_name="update_crm",
    tool_args={"record_id": "cust-42"},
    agent_role="assistant",
)

print(f"Action: {decision['action']}")
print(f"PII detected: {len(decision['pii_detected'])} items")
print()

for finding in decision["pii_detected"]:
    print(f"  Type: {finding['type']}")
    print(f"  Value: {finding['redacted']}")
    print(f"  Confidence: {finding['confidence']:.0%}")
    print()

# In OBSERVE mode, PII is flagged but not blocked
# In ENFORCE mode with PII policies, this step would be denied

## Policy Mode: Enforce Governance Rules

When you provide a `TealEngine` instance, the hook evaluates configured policies and **blocks steps** that violate governance rules. The governance evaluation is deterministic — same input always produces the same decision.

Below we demonstrate both ALLOW and DENY scenarios.

In [ ]:
from camelai_tealtiger import TealTigerAgentHook, GovernanceDenyError
from unittest.mock import MagicMock

# --- Mock TealEngine for demonstration ---
# In production, use: from tealtiger import TealEngine
mock_engine = MagicMock()

# Configure mock to ALLOW web_search but DENY delete_database
def mock_evaluate(request):
    """Simulate policy evaluation: allowlist = [web_search, summarize]"""
    allowed_tools = {"web_search", "summarize"}
    if request.get("tool_name") in allowed_tools:
        return {
            "action": "ALLOW",
            "reason": "Tool in allowlist",
            "reason_codes": [],
            "risk_score": 0,
        }
    else:
        return {
            "action": "DENY",
            "reason": f"Tool '{request.get('tool_name')}' not in allowlist",
            "reason_codes": ["TOOL_NOT_ALLOWED"],
            "risk_score": 85,
        }

mock_engine.evaluate = mock_evaluate

# Create hook with engine in ENFORCE mode
hook = TealTigerAgentHook(engine=mock_engine, mode="ENFORCE")

# --- Scenario 1: ALLOW ---
print("=" * 50)
print("Scenario 1: Allowed tool call")
print("=" * 50)

decision = hook.pre_step(
    agent_id="agent-1",
    step_content="Search for AI governance papers",
    tool_name="web_search",
    tool_args={"query": "AI governance 2025"},
    agent_role="researcher",
)
print(f"Action: {decision['action']}")
print(f"Reason: {decision['reason']}")

# --- Scenario 2: DENY ---
print(f"\n{'=' * 50}")
print("Scenario 2: Blocked tool call")
print("=" * 50)

try:
    decision = hook.pre_step(
        agent_id="agent-1",
        step_content="Delete all records from the database",
        tool_name="delete_database",
        tool_args={"table": "users"},
        agent_role="assistant",
    )
except GovernanceDenyError as e:
    print(f"🚫 Blocked!")
    print(f"   Reason: {e.decision['reason']}")
    print(f"   Codes: {e.decision['reason_codes']}")
    print(f"   Risk Score: {e.decision['risk_score']}")

## Kill Switch: Freeze & Unfreeze Agents

In multi-agent systems, one misbehaving agent shouldn't bring down the entire session. The **kill switch** lets you freeze a specific agent — all its subsequent steps are immediately blocked — while other agents continue operating normally.

In [ ]:
from camelai_tealtiger import TealTigerAgentHook, GovernanceDenyError

hook = TealTigerAgentHook(mode="ENFORCE")

# Normal operation — agent is allowed
decision = hook.pre_step(
    agent_id="rogue-agent",
    step_content="Performing normal task...",
    tool_name="summarize",
    agent_role="assistant",
)
print(f"Before freeze: {decision['action']}")  # ALLOW

# Agent acting up? Freeze it immediately
hook.freeze("rogue-agent")
print("\n❄️  Agent 'rogue-agent' is now FROZEN")

# All subsequent steps for this agent are blocked
try:
    hook.pre_step(
        agent_id="rogue-agent",
        step_content="Trying to do something...",
        tool_name="web_search",
        agent_role="assistant",
    )
except GovernanceDenyError as e:
    print(f"After freeze: DENIED — {e.decision['reason']}")

# Other agents in the same session continue normally
other_decision = hook.pre_step(
    agent_id="good-agent",
    step_content="Still working fine...",
    tool_name="summarize",
    agent_role="assistant",
)
print(f"\nOther agent: {other_decision['action']}")  # ALLOW

# Restore when ready
hook.unfreeze("rogue-agent")
print("\n🔓 Agent 'rogue-agent' is now UNFROZEN")

decision = hook.pre_step(
    agent_id="rogue-agent",
    step_content="Back to normal operations",
    tool_name="summarize",
    agent_role="assistant",
)
print(f"After unfreeze: {decision['action']}")  # ALLOW

## Role Validation

Restrict which roles can operate in your multi-agent system. Any agent attempting to act with an unauthorized role is immediately blocked.

In [ ]:
from camelai_tealtiger import TealTigerAgentHook, GovernanceDenyError

hook = TealTigerAgentHook(
    mode="ENFORCE",
    role_allowlist=["assistant", "critic", "researcher"],
)

# Allowed role — passes through
decision = hook.pre_step(
    agent_id="a1",
    step_content="Reviewing the draft...",
    tool_name="review",
    agent_role="critic",
)
print(f"Role 'critic': {decision['action']}")  # ALLOW

# Another allowed role
decision = hook.pre_step(
    agent_id="a2",
    step_content="Searching for papers...",
    tool_name="web_search",
    agent_role="researcher",
)
print(f"Role 'researcher': {decision['action']}")  # ALLOW

# Unauthorized role — raises GovernanceDenyError
try:
    hook.pre_step(
        agent_id="a3",
        step_content="Escalating privileges...",
        tool_name="admin_panel",
        agent_role="admin",
    )
except GovernanceDenyError as e:
    print(f"\nRole 'admin': DENIED")
    print(f"  Reason: {e.decision['reason']}")
    print(f"  Allowed roles: {['assistant', 'critic', 'researcher']}")

## Behavioral Baseline

After observing agent behavior in zero-config mode, generate a **behavioral baseline**. This captures normal patterns (cost per step, common tools, PII frequency) that can later be used to detect anomalies or auto-generate policies.

In [ ]:
from camelai_tealtiger import TealTigerAgentHook

hook = TealTigerAgentHook()

# Simulate several agent steps to build up history
steps = [
    ("agent-alpha", "Search for recent papers", "web_search", "researcher"),
    ("agent-alpha", "Summarize findings", "summarize", "researcher"),
    ("agent-alpha", "Draft the report with john@example.com", "write", "researcher"),
    ("agent-beta", "Review the draft", "review", "critic"),
    ("agent-beta", "Provide feedback", "comment", "critic"),
]

for agent_id, content, tool, role in steps:
    hook.pre_step(
        agent_id=agent_id,
        step_content=content,
        tool_name=tool,
        agent_role=role,
    )
    hook.post_step(
        agent_id=agent_id,
        step_result=f"Completed: {content}",
        token_usage={"prompt_tokens": 150, "completion_tokens": 100},
    )

# Generate behavioral baseline
baseline = hook.get_baseline()

print("--- Behavioral Baseline ---\n")
for agent_id, entry in baseline.items():
    print(f"{agent_id}:")
    print(f"  Steps observed: {entry.step_count}")
    print(f"  Avg cost/step: ${entry.avg_cost_per_step:.6f}")
    print(f"  Common tools: {entry.common_tools}")
    print(f"  PII frequency: {entry.pii_frequency:.2%}")
    print()

## Audit Trail & TEEC (Typed Evidence & Evidence Contract)

Every governance decision produces a structured **TEEC record** under the `teec.camelai` namespace. This provides a full audit trail compatible with compliance frameworks and observability tooling.

TEEC fields include:
- `session_id` — unique per hook instance
- `agent_id` — which agent triggered the evaluation
- `step_id` — monotonically increasing step counter
- `correlation_id` — UUID v4 for tracing across systems
- `action` — governance decision (ALLOW / DENY)
- `reason_codes` — machine-readable codes for the decision
- `risk_score` — 0-100 risk assessment
- `pii_detected` — list of PII findings
- `timestamp_iso` — ISO 8601 timestamp
- `task_prompt_hash` — SHA-256 of the task (never stored in plaintext)

In [ ]:
import json
from camelai_tealtiger import TealTigerAgentHook

hook = TealTigerAgentHook(
    task_prompt="Research AI safety and write a summary report",
    society_id="research-team-001",
)

# Run a step
hook.pre_step(
    agent_id="researcher-1",
    step_content="Search for AI safety papers from 2024-2025",
    tool_name="web_search",
    tool_args={"query": "AI safety papers 2025"},
    agent_role="researcher",
)

# Access the TEEC audit trail
audit_trail = hook.get_audit_trail()

print("--- TEEC Audit Record (teec.camelai namespace) ---\n")
for record in audit_trail:
    print(json.dumps({
        "namespace": "teec.camelai",
        "session_id": record["session_id"],
        "society_id": record["society_id"],
        "agent_id": record["agent_id"],
        "agent_role": record["agent_role"],
        "step_id": record["step_id"],
        "correlation_id": record["correlation_id"],
        "action": record["action"],
        "tool_name": record["tool_name"],
        "risk_score": record["risk_score"],
        "reason_codes": record["reason_codes"],
        "pii_detected": record["pii_detected"],
        "task_prompt_hash": record["task_prompt_hash"],
        "timestamp_iso": record["timestamp_iso"],
    }, indent=2))

print("\n--- Fields in teec.camelai namespace ---")
print("session_id:       Unique per hook instance (UUID v4)")
print("society_id:       Links agents in the same group")
print("agent_id:         Which agent triggered evaluation")
print("step_id:          Monotonically increasing counter")
print("correlation_id:   OpenTelemetry-compatible trace ID")
print("action:           ALLOW | DENY")
print("risk_score:       0-100 assessment")
print("task_prompt_hash: SHA-256 (prompt never stored in plaintext)")

## Summary

| Mode | Behavior | Use Case |
|------|----------|----------|
| `OBSERVE` | Allow all, track cost, detect PII, build baseline | Zero-config default — start here |
| `MONITOR` | Evaluate policies, log decisions, allow all | Staging — see what *would* be blocked |
| `ENFORCE` | Block steps that violate policies | Production — hard enforcement |

| Feature | Description |
|---------|-------------|
| Zero-Config | Observe, track cost, detect PII with no setup |
| Policy Enforcement | Block unauthorized tools, roles, and actions |
| PII Detection | In-process scanning for emails, SSNs, phone numbers |
| Kill Switch | Freeze/unfreeze individual agents mid-conversation |
| Role Validation | Allowlist which roles can operate |
| Behavioral Baseline | Capture normal patterns for anomaly detection |
| TEEC Audit Trail | Structured evidence with correlation IDs |

## Key Properties

- **Deterministic:** No LLM in the governance path — same input always produces the same decision
- **Fast:** <2ms per evaluation, in-process with zero external calls
- **Non-invasive:** Works as a hook — no changes to your existing CAMEL agent code
- **Granular:** Per-agent control (freeze one agent, not all)

## Resources

- [TealTiger GitHub](https://github.com/agentguard-ai/tealtiger) — Apache 2.0
- [PyPI: camelai-tealtiger](https://pypi.org/project/camelai-tealtiger/)
- [TealTiger Documentation](https://docs.tealtiger.ai)
- [CAMEL-AI Framework](https://github.com/camel-ai/camel)
- [OWASP Agentic Security Index](https://owasp.org/www-project-agentic-security-initiative/) — TealTiger covers 8/10 categories